In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [ ]:
import os
import time
import yfinance as yf
import pandas as pd

TICKERS = [
    "RELIANCE.NS","TCS.NS","HDFCBANK.NS","ICICIBANK.NS","INFY.NS","HINDUNILVR.NS","ITC.NS","SBIN.NS",
    "BHARTIARTL.NS","KOTAKBANK.NS","LT.NS","AXISBANK.NS","BAJFINANCE.NS","ASIANPAINT.NS","MARUTI.NS",
    "HCLTECH.NS","SUNPHARMA.NS","TITAN.NS","ULTRACEMCO.NS","NESTLEIND.NS","WIPRO.NS","ONGC.NS","NTPC.NS",
    "POWERGRID.NS","M&M.NS","TATAMOTORS.NS","TATASTEEL.NS","JSWSTEEL.NS","ADANIENT.NS","ADANIPORTS.NS",
    "COALINDIA.NS","BAJAJFINSV.NS","HDFCLIFE.NS","SBILIFE.NS","DIVISLAB.NS","DRREDDY.NS","CIPLA.NS",
    "GRASIM.NS","BRITANNIA.NS","EICHERMOT.NS","HEROMOTOCO.NS","BPCL.NS","HINDALCO.NS","TECHM.NS",
    "INDUSINDBK.NS","APOLLOHOSP.NS","BAJAJ-AUTO.NS","UPL.NS","LTIM.NS","TATACONSUM.NS","SHRIRAMFIN.NS"
]

START = "2010-01-01"
END = None

DATA_DIR = "nifty50_data"
os.makedirs(DATA_DIR, exist_ok=True)

raw = {}
failed = []

for ticker in TICKERS:

    success = False

    for attempt in range(3):

        try:

            df = yf.download(
                ticker,
                start=START,
                end=END,
                auto_adjust=True,
                progress=False,
                threads=False,
                timeout=60,
            )

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            if not df.empty:

                raw[ticker] = df
                success = True
                print(f"✓ {ticker:<18} {len(df):>5} rows")
                break

        except Exception as e:
            print(f"{ticker} | Attempt {attempt+1}/3 | {e}")

        time.sleep(2)

    if not success:
        failed.append(ticker)

print("\n================================")
print(f"Downloaded : {len(raw)}")
print(f"Failed     : {len(failed)}")

if failed:
    print("\nFailed tickers:")
    print(failed)

Failed to get ticker 'RELIANCE.NS' reason: Failed to perform, curl: (28) Resolving timed out after 30000 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

1 Failed download:
['RELIANCE.NS']: Timeout('Failed to perform, curl: (28) Resolving timed out after 30001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
Failed to get ticker 'RELIANCE.NS' reason: Failed to perform, curl: (28) Resolving timed out after 30000 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

1 Failed download:
['RELIANCE.NS']: Timeout('Failed to perform, curl: (28) Resolving timed out after 30001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
Failed to get ticker 'RELIANCE.NS' reason: Failed to perform, curl: (28) Resolving timed out after 30001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

1 Failed download:
['REL

In [ ]:
# Save processed dataset
SAVE_PATH = "/kaggle/working/nifty50_data.parquet"
import pandas as pd

data = pd.DataFrame(raw)
data.to_parquet("/kaggle/working/data.parquet")
data.to_parquet(SAVE_PATH, compression="snappy", index=True)

print(f"✓ Saved: {SAVE_PATH}")
print(f"Shape: {data.shape}")

In [ ]:
for t, d in raw.items():
    d.to_csv(f"{DATA_DIR}/{t.replace('.', '_')}.csv")

In [ ]:
data_by_ticker = {}
for t in TICKERS:
    path = f"{DATA_DIR}/{t.replace('.', '_')}.csv"
    if os.path.exists(path):
        d = pd.read_csv(path, index_col=0, parse_dates=True)
        data_by_ticker[t] = d.sort_index()

In [ ]:
SECTOR_MAP = {
    "RELIANCE.NS":"Energy","TCS.NS":"IT","HDFCBANK.NS":"Banking","ICICIBANK.NS":"Banking","INFY.NS":"IT",
    "HINDUNILVR.NS":"FMCG","ITC.NS":"FMCG","SBIN.NS":"Banking","BHARTIARTL.NS":"Telecom","KOTAKBANK.NS":"Banking",
    "LT.NS":"Infrastructure","AXISBANK.NS":"Banking","BAJFINANCE.NS":"Financial Services","ASIANPAINT.NS":"Consumer Durables",
    "MARUTI.NS":"Auto","HCLTECH.NS":"IT","SUNPHARMA.NS":"Pharma","TITAN.NS":"Consumer Durables","ULTRACEMCO.NS":"Cement",
    "NESTLEIND.NS":"FMCG","WIPRO.NS":"IT","ONGC.NS":"Energy","NTPC.NS":"Power","POWERGRID.NS":"Power","M&M.NS":"Auto",
    "TATAMOTORS.NS":"Auto","TATASTEEL.NS":"Metals","JSWSTEEL.NS":"Metals","ADANIENT.NS":"Conglomerate","ADANIPORTS.NS":"Infrastructure",
    "COALINDIA.NS":"Energy","BAJAJFINSV.NS":"Financial Services","HDFCLIFE.NS":"Insurance","SBILIFE.NS":"Insurance",
    "DIVISLAB.NS":"Pharma","DRREDDY.NS":"Pharma","CIPLA.NS":"Pharma","GRASIM.NS":"Cement","BRITANNIA.NS":"FMCG",
    "EICHERMOT.NS":"Auto","HEROMOTOCO.NS":"Auto","BPCL.NS":"Energy","HINDALCO.NS":"Metals","TECHM.NS":"IT",
    "INDUSINDBK.NS":"Banking","APOLLOHOSP.NS":"Healthcare","BAJAJ-AUTO.NS":"Auto","UPL.NS":"Chemicals",
    "LTIM.NS":"IT","TATACONSUM.NS":"FMCG","SHRIRAMFIN.NS":"Financial Services"
}
MARKET_CAP_MAP = {t: "Large" for t in TICKERS}

sector_ids = {s: i for i, s in enumerate(sorted(set(SECTOR_MAP.values())))}
cap_ids = {"Large": 0, "Mid": 1, "Small": 2}
ticker_ids = {t: i for i, t in enumerate(TICKERS)}

feat_by_ticker = {}
for t, df in data_by_ticker.items():
    o, h, l, c, v = df['Open'], df['High'], df['Low'], df['Close'], df['Volume']
    #ret1 = c.pct_change()

    feat = pd.DataFrame(index=df.index)

    feat['mom_5'] = c.pct_change(5)
    feat['mom_10'] = c.pct_change(10)
    feat['mom_20'] = c.pct_change(20)
    feat['mom_60'] = c.pct_change(60)

    sma20 = c.rolling(20).mean()
    sma50 = c.rolling(50).mean()
    std20 = c.rolling(20).std()

    feat['sma20_dist'] = c/sma20 - 1
    feat['sma50_dist'] = c/sma50 - 1
    feat['sma_slope_5'] = sma20.pct_change(5)
    feat['trend_slope20'] = c.rolling(20).apply(lambda x: np.polyfit(np.arange(len(x)), x, 1)[0]/x.mean(), raw=True)

    vol10 = ret1.rolling(10).std()
    vol30 = ret1.rolling(30).std()
    feat['vol_10'] = vol10
    feat['vol_30'] = vol30
    feat['vol_regime'] = vol10/vol30

    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    feat['atr14'] = tr.rolling(14).mean()/c

    bb_upper = sma20 + 2*std20
    bb_lower = sma20 - 2*std20
    bb_width = (bb_upper - bb_lower)/sma20
    feat['bb_width'] = bb_width
    feat['bb_width_pctile'] = bb_width.rolling(100).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)

    feat['donchian_width20'] = (h.rolling(20).max() - l.rolling(20).min())/c

    feat['dist_high_252'] = c/h.rolling(252).max() - 1
    feat['dist_low_252'] = c/l.rolling(252).min() - 1

    ll14 = l.rolling(14).min()
    hh14 = h.rolling(14).max()
    feat['stoch_k14'] = (c - ll14)/(hh14 - ll14)

    delta = c.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain/loss
    feat['rsi14'] = 100 - 100/(1+rs)

    feat['mom_consistency_10'] = (ret1 > 0).rolling(10).mean()
    feat['sharpe_mom_20'] = feat['mom_20']/(vol10*np.sqrt(20) + 1e-9)

    feat['vol_ratio'] = v/v.rolling(20).mean()
    obv = (np.sign(ret1).fillna(0)*v).cumsum()
    feat['obv_slope_10'] = obv.pct_change(10)
    feat['pv_corr20'] = ret1.rolling(20).corr(v.pct_change())

    feat['z_score20'] = (c - sma20)/std20

    feat['autocorr_ret5'] = ret1.rolling(20).apply(lambda x: pd.Series(x).autocorr(lag=1), raw=False)

    feat['mom_vol_interaction'] = feat['mom_10']*feat['vol_regime']
    feat['volume_confirmed_move'] = ret1*feat['vol_ratio']

    #feat['ret1'] = ret1

    feat['Ticker_ID'] = ticker_ids[t]
    feat['Sector'] = sector_ids[SECTOR_MAP[t]]
    feat['Market_Cap_Category'] = cap_ids[MARKET_CAP_MAP[t]]

    feat_by_ticker[t] = feat

In [ ]:
nifty = yf.download("^NSEI", start=START, end=END, auto_adjust=True)
if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = nifty.columns.get_level_values(0)
nifty = nifty.sort_index()

n_o, n_h, n_l, n_c, n_v = nifty['Open'], nifty['High'], nifty['Low'], nifty['Close'], nifty['Volume']
n_ret1 = n_c.pct_change()

regime = pd.DataFrame(index=nifty.index)
regime['nifty_return'] = n_ret1

n_ema20 = n_c.ewm(span=20, adjust=False).mean()
n_ema50 = n_c.ewm(span=50, adjust=False).mean()
regime['nifty_ema20_dist'] = n_c/n_ema20 - 1
regime['nifty_ema50_dist'] = n_c/n_ema50 - 1

n_tr = pd.concat([n_h-n_l, (n_h-n_c.shift()).abs(), (n_l-n_c.shift()).abs()], axis=1).max(axis=1)
n_atr14 = n_tr.rolling(14).mean()
regime['nifty_atr14'] = n_atr14/n_c
regime['nifty_atr_ratio'] = n_atr14/n_atr14.rolling(50).mean()

n_delta = n_c.diff()
n_gain = n_delta.clip(lower=0).rolling(14).mean()
n_loss = (-n_delta.clip(upper=0)).rolling(14).mean()
n_rs = n_gain/n_loss
regime['nifty_rsi14'] = 100 - 100/(1+n_rs)

n_sma20 = n_c.rolling(20).mean()
n_std20 = n_c.rolling(20).std()
regime['nifty_bb_width'] = (4*n_std20)/n_sma20

regime['nifty_volatility'] = n_ret1.rolling(20).std()
regime['nifty_momentum'] = n_c.pct_change(20)
regime['nifty_trend_slope'] = n_c.rolling(20).apply(lambda x: np.polyfit(np.arange(len(x)), x, 1)[0]/x.mean(), raw=True)

for t in feat_by_ticker:
    feat_by_ticker[t] = feat_by_ticker[t].join(regime, how='left')

In [ ]:
sector_frames = []
for t, feat in feat_by_ticker.items():
    tmp = pd.DataFrame({
        'Sector_Name': SECTOR_MAP[t],
        'ret1': data_by_ticker[t]['Close'].pct_change(),
        'mom_10': feat['mom_10'],
        'atr14': feat['atr14'],
        'rsi14': feat['rsi14'],
        'sma20_dist': feat['sma20_dist'],
        'vol_ratio': feat['vol_ratio'],
    }, index=feat.index)
    sector_frames.append(tmp)

sector_panel = pd.concat(sector_frames, axis=0)
sector_panel.index.name = 'Date'
sector_panel = sector_panel.reset_index()

sector_agg = sector_panel.groupby(['Sector_Name', 'Date']).mean(numeric_only=True)
sector_agg = sector_agg.rename(columns={
    'ret1': 'sector_avg_return',
    'mom_10': 'sector_momentum',
    'atr14': 'sector_atr',
    'rsi14': 'sector_rsi',
    'sma20_dist': 'sector_ema_dist',
    'vol_ratio': 'sector_volume_ratio',
})

for t, feat in feat_by_ticker.items():
    sec_data = sector_agg.xs(SECTOR_MAP[t], level='Sector_Name')
    feat_by_ticker[t] = feat.join(sec_data, how='left')
    feat_by_ticker[t]['relative_strength_nifty'] = feat_by_ticker[t]['sector_avg_return'] - feat_by_ticker[t]['nifty_return']

In [ ]:
# Adjust desired return threshold here
TARGET_RETURN = 0.05
MODE = "day"   # "week" or "day"

data_list = []
for t, feat in feat_by_ticker.items():
    df = data_by_ticker[t]
    o, c = df['Open'], df['Close']

    if MODE == "week":
        week_period = df.index.to_period('W-FRI')
        is_monday = df.index.dayofweek == 0
        is_friday = df.index.dayofweek == 4

        monday_open = o[is_monday].groupby(week_period[is_monday]).first()
        friday_close = c[is_friday].groupby(week_period[is_friday]).first()

        weekly = pd.concat([monday_open.rename('mon_open'), friday_close.rename('fri_close')], axis=1).dropna()
        weekly['target'] = (weekly['fri_close'] >= weekly['mon_open'] * (1 + TARGET_RETURN)).astype(int)

        target = pd.Series(index=df.index, dtype='float64')
        target.loc[df.index[is_monday]] = week_period[is_monday].map(weekly['target'])
    else:
        next_close = c.shift(-1)
        target = (next_close >= o * (1 + TARGET_RETURN)).astype(int).astype('float64')
        target[next_close.isna()] = np.nan

    d = feat.copy()
    d['target'] = target
    d = d.dropna()
    d['__ticker__'] = t
    data_list.append(d)

data = pd.concat(data_list, axis=0)

assert set(data['target'].unique()).issubset({0, 1})

In [ ]:
train_list, test_list = [], []
for t in data['__ticker__'].unique():
    sub = data[data['__ticker__'] == t]
    split_idx = int(len(sub) * 0.8)
    train_list.append(sub.iloc[:split_idx])
    test_list.append(sub.iloc[split_idx:])

train = pd.concat(train_list, axis=0).drop(columns='__ticker__')
test = pd.concat(test_list, axis=0).drop(columns='__ticker__')

X_train, y_train = train.drop(columns='target'), train['target']
X_test, y_test = test.drop(columns='target'), test['target']

In [ ]:
val_frac = 0.15
train_sub_list, val_list = [], []
for sub in train_list:
    split_idx = int(len(sub) * (1 - val_frac))
    train_sub_list.append(sub.iloc[:split_idx])
    val_list.append(sub.iloc[split_idx:])

fwd_frac = 0.5
test_sub_list, forward_list = [], []
for sub in test_list:
    split_idx = int(len(sub) * (1 - fwd_frac))
    test_sub_list.append(sub.iloc[:split_idx])
    forward_list.append(sub.iloc[split_idx:])

train = pd.concat(train_sub_list, axis=0).drop(columns='__ticker__')
val = pd.concat(val_list, axis=0).drop(columns='__ticker__')
test = pd.concat(test_sub_list, axis=0).drop(columns='__ticker__')
forward = pd.concat(forward_list, axis=0).drop(columns='__ticker__')

X_train, y_train = train.drop(columns='target'), train['target']
X_val, y_val = val.drop(columns='target'), val['target']
X_test, y_test = test.drop(columns='target'), test['target']
X_forward, y_forward = forward.drop(columns='target'), forward['target']

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

In [ ]:
val_proba = model.predict_proba(X_val)[:,1]
thresholds = np.round(np.arange(0.10, 0.901, 0.05), 2)
best_threshold, best_f1 = 0.5, -1
for thr in thresholds:
    f1 = f1_score(y_val, (val_proba >= thr).astype(int))
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

test_proba = model.predict_proba(X_test)[:,1]
test_pred = (test_proba >= best_threshold).astype(int)

forward_proba = model.predict_proba(X_forward)[:,1]
forward_pred = (forward_proba >= best_threshold).astype(int)

print("Best Threshold:", best_threshold)
print("Validation F1:", best_f1)
print("Forward F1:", f1_score(y_forward, forward_pred))
print("Forward Precision:", precision_score(y_forward, forward_pred))
print("Forward Recall:", recall_score(y_forward, forward_pred))